# Proyecto 1 — Simulador de Máquina de Turing (1 cinta) para Fibonacci + análisis empírico

Este notebook contiene **todo** lo necesario para:

1. Cargar una Máquina de Turing determinista de **una cinta** desde un archivo `maquina_turing.json`.
2. Simular su ejecución paso a paso, mostrando **configuraciones**: (estado, posición de la cabeza, cinta).
3. Ejecutar un **análisis empírico** del tiempo de ejecución y número de pasos (transiciones):
   - **(5.a)** listar entradas de prueba,
   - **(5.b)** diagrama de dispersión tiempo vs tamaño de entrada,
   - **(5.c)** regresión polinomial y selección del mejor grado por **R²**.

> **Convención usada**  
> Entrada: `n` en unario con `'1'` (n = 1^n). Para n=0, la entrada es vacía.  
> Salida: separador `'#'` y a la derecha `0^F(n)`.

**Archivos requeridos en el mismo directorio del notebook:**
- `maquina_turing.json`


In [ ]:
# Si estás en un entorno limpio, puedes instalar dependencias.
# En la mayoría de entornos de Jupyter ya vienen instaladas.
%pip -q install numpy matplotlib pandas

import json
import time
import math
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

## 1) Recordatorio rápido: Máquina de Turing determinista (1 cinta)

Una Máquina de Turing (TM) se define por:

- **Estados**: conjunto `Q`
- **Alfabeto de entrada**: `Σ` (lo que viene inicialmente)
- **Alfabeto de cinta**: `Γ` (incluye símbolos auxiliares y el blanco `_`)
- **Función de transición**: `δ(q, s) = (q', s', movimiento)` con movimiento `L` o `R`
- **Estado inicial**: `q0`
- **Estados de parada**: `q_accept`, `q_reject`

En una TM **determinista**, para cada par (estado, símbolo leído) hay **a lo sumo una** transición posible.


In [ ]:
# ---------------------------
# Utilidades matemáticas
# ---------------------------

def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Coeficiente de determinación R²."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (ss_res / ss_tot) if ss_tot != 0 else 0.0


def train_test_split_indices(n: int, test_ratio: float = 0.3, seed: int = 42):
    """Devuelve índices (train_idx, test_idx) para dividir datos."""
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    test_size = max(1, int(round(n * test_ratio)))
    test_idx = idx[:test_size]
    train_idx = idx[test_size:]
    return train_idx, test_idx


def best_polynomial_fit(x: np.ndarray, y: np.ndarray, degrees=range(1, 7), test_ratio: float = 0.3, seed: int = 42):
    """Prueba varios grados polinomiales y elige el mejor por R² en test."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    train_idx, test_idx = train_test_split_indices(len(x), test_ratio=test_ratio, seed=seed)
    x_train, y_train = x[train_idx], y[train_idx]
    x_test, y_test = x[test_idx], y[test_idx]

    results = []
    for d in degrees:
        coeffs = np.polyfit(x_train, y_train, deg=d)
        p = np.poly1d(coeffs)
        y_pred_test = p(x_test)
        score = r2_score(y_test, y_pred_test)
        results.append((d, score, coeffs))

    # Ordenar por mejor R²
    results.sort(key=lambda t: t[1], reverse=True)
    best_degree, best_r2, best_coeffs = results[0]
    return {
        "best_degree": best_degree,
        "best_r2": float(best_r2),
        "best_coeffs": best_coeffs,
        "all": results
    }


def poly_equation_string(coeffs: np.ndarray, var: str = "n") -> str:
    """Devuelve una ecuación bonita tipo a*n^k + ..."""
    coeffs = np.asarray(coeffs, dtype=float)
    deg = len(coeffs) - 1
    parts = []
    for i, a in enumerate(coeffs):
        power = deg - i
        if abs(a) < 1e-12:
            continue
        if power == 0:
            parts.append(f"{a:.3e}")
        elif power == 1:
            parts.append(f"{a:.3e}·{var}")
        else:
            parts.append(f"{a:.3e}·{var}^{power}")
    return " + ".join(parts) if parts else "0"

## 2) Simulador de Máquina de Turing (cinta infinita con diccionario)

### Representación
- La **cinta** se modela como un diccionario `cinta[pos] = símbolo`.
- Si una celda no existe en el diccionario, se asume que contiene el blanco `_`.
- La **cabeza** es un entero `pos`.
- El **estado** es una cadena, por ejemplo `q0`, `init_find_end`, etc.

### Configuración
Una “configuración” es:  
\[
(	ext{estado actual},\ 	ext{posición de la cabeza},\ 	ext{contenido relevante de la cinta})
\]


In [ ]:
@dataclass
class ResultadoSimulacion:
    aceptada: bool
    estado_final: str
    pasos: int
    cinta: Dict[int, str]
    cabeza: int


class MaquinaTuring:
    def __init__(self, ruta_json: str):
        with open(ruta_json, 'r', encoding='utf-8') as f:
            config = json.load(f)

        self.estados = config['estados']
        self.alfabeto_entrada = set(config['alfabeto_entrada'])
        self.alfabeto_cinta = set(config['alfabeto_cinta'])

        self.estado_inicial = config['estado_inicial']
        self.estado_aceptacion = config['estado_aceptacion']
        self.estado_rechazo = config['estado_rechazo']
        self.transiciones = config['transiciones']

        # Símbolo blanco por convención del JSON
        self.blanco = '_'

    def _leer(self, cinta: Dict[int, str], pos: int) -> str:
        return cinta.get(pos, self.blanco)

    def _escribir(self, cinta: Dict[int, str], pos: int, simbolo: str):
        if simbolo == self.blanco:
            # Para mantener el diccionario pequeño, podemos borrar blancos explícitos
            cinta.pop(pos, None)
        else:
            cinta[pos] = simbolo

    def _rango_util(self, cinta: Dict[int, str], cabeza: int):
        if not cinta:
            return cabeza, cabeza
        min_idx = min(min(cinta.keys()), cabeza)
        max_idx = max(max(cinta.keys()), cabeza)
        return min_idx, max_idx

    def _formatear_cinta(self, cinta: Dict[int, str], cabeza: int) -> str:
        min_idx, max_idx = self._rango_util(cinta, cabeza)
        piezas = []
        for j in range(min_idx, max_idx + 1):
            s = self._leer(cinta, j)
            if j == cabeza:
                piezas.append(f"[{s}]")
            else:
                piezas.append(s)
        return " ".join(piezas)

    def simular(self, entrada: str, verbose: bool = True, max_pasos: int = 2_000_000) -> ResultadoSimulacion:
        """Simula la TM hasta aceptar/rechazar o exceder max_pasos.

        - entrada: cadena inicial (por convención, solo '1' para n unario).
        - verbose: imprime configuraciones.
        - max_pasos: guardia para evitar bucles infinitos accidentales.
        """
        # Validación mínima de entrada según Σ
        for ch in entrada:
            if ch not in self.alfabeto_entrada:
                raise ValueError(f"Símbolo inválido en la entrada: {ch!r}. Alfabeto de entrada permitido: {sorted(self.alfabeto_entrada)}")

        # Cinta infinita con diccionario
        cinta: Dict[int, str] = {i: c for i, c in enumerate(entrada)}
        estado = self.estado_inicial
        cabeza = 0
        pasos = 0

        if verbose:
            print(f"Entrada: {entrada!r}")
            print("=" * 70)

        while estado not in (self.estado_aceptacion, self.estado_rechazo):
            simbolo = self._leer(cinta, cabeza)

            if verbose:
                cinta_str = self._formatear_cinta(cinta, cabeza)
                print(f"Paso {pasos:6d} | Estado={estado:15s} | Cabeza={cabeza:6d} | Cinta: {cinta_str}")

            # Buscar transición
            try:
                nuevo_estado, nuevo_simbolo, direccion = self.transiciones[estado][simbolo]
            except KeyError:
                # Si no hay transición, formalmente podemos considerar rechazo
                estado = self.estado_rechazo
                break

            # Aplicar transición: escribir, cambiar estado, mover cabeza
            self._escribir(cinta, cabeza, nuevo_simbolo)
            estado = nuevo_estado

            if direccion == 'R':
                cabeza += 1
            elif direccion == 'L':
                cabeza -= 1
            else:
                raise ValueError(f"Dirección inválida en transición: {direccion!r}")

            pasos += 1
            if pasos > max_pasos:
                estado = self.estado_rechazo
                break

        aceptada = (estado == self.estado_aceptacion)

        if verbose:
            print("=" * 70)
            print(f"HALT: estado_final={estado}, aceptada={aceptada}, pasos={pasos}")
            print(f"Cinta final: {self._formatear_cinta(cinta, cabeza)}")

        return ResultadoSimulacion(
            aceptada=aceptada,
            estado_final=estado,
            pasos=pasos,
            cinta=cinta,
            cabeza=cabeza
        )


def extraer_salida_fibonacci(cinta: Dict[int, str]) -> int:
    """Interpreta la cinta final con la convención: '# 0 0 0 ...' => cuenta 0s a la derecha.

    Devuelve F(n) (cantidad de '0' a la derecha del primer '#').
    Si no encuentra '#', devuelve 0.
    """
    if not cinta:
        return 0

    min_idx = min(cinta.keys())
    max_idx = max(cinta.keys())

    # Construir string "denso" en rango útil
    s = ''.join(cinta.get(i, '_') for i in range(min_idx, max_idx + 1))

    if '#' not in s:
        return 0

    right = s.split('#', 1)[1]
    # contar ceros hasta que aparezca un símbolo distinto (o termine)
    count = 0
    for ch in right:
        if ch == '0':
            count += 1
        elif ch == '_':
            break
        else:
            # si hay marcadores, dejamos de contar (idealmente ya no hay)
            break
    return count

## 3) Simulación paso a paso (requisito 4.b y 4.c)

- Ingresa una cadena unaria:  
  - ejemplo: `111` representa n=3  
  - ejemplo: *(vacío)* representa n=0
- El simulador imprimirá configuraciones indicando:
  - **Estado**
  - **Posición de la cabeza**
  - **Cinta** (marcando la celda bajo la cabeza con `[ ]`)


In [ ]:
# Asegúrate de que el archivo JSON esté disponible.
# En este entorno, el archivo está en la misma carpeta del notebook.
ruta_json = "maquina_turing.json"

mt = MaquinaTuring(ruta_json)

# Puedes cambiar esto por una cadena fija si no quieres usar input():
entrada_str = input("Ingrese la cadena de entrada unaria (ej: 111 para n=3; vacío para n=0): ")

resultado = mt.simular(entrada_str, verbose=True, max_pasos=2_000_000)
fn = extraer_salida_fibonacci(resultado.cinta)
print(f"Interpretación de salida: F(n) = {fn}")

## 4) Análisis empírico (Entregable 5)

### Qué mediremos
Para cada `n` en un rango de pruebas:

- **Entrada**: `1^n`
- **Tamaño de entrada**: `n` (longitud de la cadena unaria)
- **Tiempo**: segundos (medido con `time.perf_counter()`)
- **Pasos**: número de transiciones ejecutadas por la TM
- **Resultado**: valor Fibonacci interpretado a partir de la cinta final

### Notas importantes
- El tiempo (segundos) depende del hardware/carga del sistema; por eso también medimos **pasos**.
- Para evitar ejecuciones demasiado largas, ajusta `n_max`.


In [ ]:
def generar_entradas_prueba(n_min: int = 1, n_max: int = 12):
    """Devuelve una lista de (n, entrada_unaria)."""
    entradas = []
    for n in range(n_min, n_max + 1):
        entradas.append((n, "1" * n))
    return entradas


def medir_rendimiento(mt: MaquinaTuring, entradas, max_pasos: int = 2_000_000):
    """Ejecuta la TM para cada entrada, midiendo tiempo y pasos."""
    filas = []
    for n, entrada in entradas:
        t0 = time.perf_counter()
        res = mt.simular(entrada, verbose=False, max_pasos=max_pasos)
        t1 = time.perf_counter()

        fn = extraer_salida_fibonacci(res.cinta)

        filas.append({
            "n": n,
            "entrada": entrada,
            "tam_entrada": len(entrada),
            "aceptada": res.aceptada,
            "pasos": res.pasos,
            "tiempo_seg": t1 - t0,
            "F(n)_salida": fn
        })
    return pd.DataFrame(filas)


# ------------- (5.a) Listado de entradas -------------
n_min, n_max = 1, 12   # ajusta si tu máquina corre más (p.ej. 15)
entradas = generar_entradas_prueba(n_min, n_max)

df_entradas = pd.DataFrame({
    "n": [n for n, _ in entradas],
    "entrada_unaria": [s for _, s in entradas],
    "tam_entrada": [len(s) for _, s in entradas]
})

print("Listado de entradas de prueba (Entregable 5.a):")
display(df_entradas)

In [ ]:
# Ejecutar mediciones
df = medir_rendimiento(mt, entradas, max_pasos=2_000_000)

print("Resultados de medición:")
display(df)

# Validación rápida: que todas hayan aceptado
if not df["aceptada"].all():
    print("⚠️ Atención: alguna prueba NO fue aceptada. Revisa 'aceptada' y/o aumenta max_pasos.")

### (5.b) Diagrama de dispersión: tiempo vs tamaño de entrada

Aquí el “tamaño de entrada” se toma como `n` (longitud de la entrada unaria).


In [ ]:
# Tiempo vs n
plt.figure(figsize=(8, 5))
plt.scatter(df["n"], df["tiempo_seg"])
plt.title("Diagrama de dispersión: Tiempo vs n")
plt.xlabel("n (tamaño de la entrada unaria)")
plt.ylabel("Tiempo (segundos)")
plt.grid(True)
plt.show()

# Pasos vs n (muy útil porque es más estable que el tiempo)
plt.figure(figsize=(8, 5))
plt.scatter(df["n"], df["pasos"])
plt.title("Diagrama de dispersión: Pasos vs n")
plt.xlabel("n (tamaño de la entrada unaria)")
plt.ylabel("Pasos (número de transiciones)")
plt.grid(True)
plt.show()

### (5.c) Regresión polinomial: elegir el mejor grado por R²

Para ajustarnos al enunciado:
- Probamos varios grados (1..6)  
- Elegimos el **mejor** usando **R² en un conjunto de prueba** (train/test split) para reducir sobreajuste.


In [ ]:
# Elegir el mejor polinomio para TIEMPO vs n
x = df["n"].to_numpy()
y_time = df["tiempo_seg"].to_numpy()

fit_time = best_polynomial_fit(x, y_time, degrees=range(1, 7), test_ratio=0.3, seed=42)
deg_time = fit_time["best_degree"]
coeffs_time = fit_time["best_coeffs"]
eq_time = poly_equation_string(coeffs_time, var="n")

print("Mejor regresión polinomial para TIEMPO vs n")
print(f"  Grado seleccionado: {deg_time}")
print(f"  R² (en test):       {fit_time['best_r2']:.4f}")
print(f"  Ecuación aprox:     T(n) ≈ {eq_time}")

# Graficar ajuste (usando todos los datos para visualizar)
p_time = np.poly1d(np.polyfit(x, y_time, deg=deg_time))
x_line = np.linspace(x.min(), x.max(), 200)
y_line = p_time(x_line)

plt.figure(figsize=(8, 5))
plt.scatter(x, y_time, label="Datos")
plt.plot(x_line, y_line, label=f"Ajuste grado {deg_time}")
plt.title("Ajuste polinomial: Tiempo vs n")
plt.xlabel("n")
plt.ylabel("Tiempo (segundos)")
plt.grid(True)
plt.legend()
plt.show()


# Repetimos lo mismo para PASOS vs n (a veces es más limpio)
y_steps = df["pasos"].to_numpy()

fit_steps = best_polynomial_fit(x, y_steps, degrees=range(1, 7), test_ratio=0.3, seed=42)
deg_steps = fit_steps["best_degree"]
coeffs_steps = fit_steps["best_coeffs"]
eq_steps = poly_equation_string(coeffs_steps, var="n")

print("Mejor regresión polinomial para PASOS vs n")
print(f"  Grado seleccionado: {deg_steps}")
print(f"  R² (en test):       {fit_steps['best_r2']:.4f}")
print(f"  Ecuación aprox:     S(n) ≈ {eq_steps}")

p_steps = np.poly1d(np.polyfit(x, y_steps, deg=deg_steps))
y_line2 = p_steps(x_line)

plt.figure(figsize=(8, 5))
plt.scatter(x, y_steps, label="Datos")
plt.plot(x_line, y_line2, label=f"Ajuste grado {deg_steps}")
plt.title("Ajuste polinomial: Pasos vs n")
plt.xlabel("n")
plt.ylabel("Pasos")
plt.grid(True)
plt.legend()
plt.show()

## 5) Expresión asintótica (lo que escribirías en el informe)

**Empírico (según regresión polinomial):**  
Si el mejor ajuste para pasos es un polinomio de grado `k`, entonces en el rango medido podemos reportar:

\[
S(n) \approx a\,n^k \Rightarrow S(n) \in O(n^k)
\]

y de manera similar para el tiempo:

\[
T(n) \in O(n^k) \quad \text{(en el rango medido)}
\]

**Importante para decir en clase / documento:**  
Esta notación basada en regresión describe **el rango medido**. Teóricamente, en esta construcción en unario, el tamaño de la salida es \(F(n)\), y la máquina realiza múltiples recorridos/copias sobre la cinta; por ello, el número de pasos crece muy rápido conforme aumenta `n`.

> Para el entregable, usualmente basta con reportar la \(O(n^k)\) empírica del rango medido **y** mencionar la intuición teórica (salida en unario + copias/re-recorridos).


## (Opcional) Comparación con funciones derivadas de Fibonacci

Si quieres explorar una hipótesis típica para TM de 1 cinta en unario, puedes comparar los datos contra \(F(n)\) o \((F(n))^2\).
Esto NO sustituye el requisito de “tiempo vs tamaño de entrada”, pero puede servir como discusión.


In [ ]:
def fibonacci(n: int) -> int:
    if n <= 0: 
        return 0
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

df["F(n)_teorico"] = df["n"].apply(fibonacci)
df["F(n)^2"] = df["F(n)_teorico"] ** 2

plt.figure(figsize=(8, 5))
plt.scatter(df["F(n)^2"], df["pasos"])
plt.title("Exploración: Pasos vs (F(n))^2")
plt.xlabel("(F(n))^2")
plt.ylabel("Pasos")
plt.grid(True)
plt.show()

display(df[["n","pasos","F(n)_teorico","F(n)^2"]])